# PatchTST Fundamental Classifier

Trains a PatchTST model on quarterly Compustat fundamentals to predict
next-quarter stock direction (down / flat / up) for S&P 500 tickers.

**Pipeline:**
1. Load `quarterly_fundamentals.csv` → `patchtst_lib.fundamental.features`
2. Load `historic_data_rows.csv` → price-lookup for labels
3. Chronological train / val / test split (≤ 2019Q4 / 2020-2021 / 2022+)
4. `QuarterlyFundamentalDataset` with 45-day publish-lag and fixed-pct labels
5. `PatchTSTClassifier(horizon=1)` with `make_fundamental_patchtst_config()`
6. HuggingFace Trainer with early stopping on val macro-F1
7. Save checkpoint + metadata

In [1]:
import platform
from pathlib import Path

import torch

DEVICE_OVERRIDE = None   # None | 'cuda' | 'mps' | 'cpu'
DTYPE_OVERRIDE  = 'float32'   # None | 'bfloat16' | 'float16' | 'float32'


def _autoselect_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def _autoselect_dtype(device):
    if device.type == 'cuda':
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


device = torch.device(DEVICE_OVERRIDE) if DEVICE_OVERRIDE else _autoselect_device()
dtype  = getattr(torch, DTYPE_OVERRIDE) if DTYPE_OVERRIDE else _autoselect_dtype(device)
NUM_WORKERS = 0 if platform.system() == 'Windows' else 2

print(f'OS={platform.system()} | device={device} | dtype={dtype} | workers={NUM_WORKERS}')


OS=Linux | device=cuda | dtype=torch.float32 | workers=2


In [2]:
import subprocess
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer

# ===== Environment Bootstrap =====
IS_KAGGLE = Path('/kaggle/input').exists()

if IS_KAGGLE:
    REPO_URL    = 'https://github.com/AntonyAPT/SeniorProject.git'
    REPO_BRANCH = 'feature/zamnz_patchTST_weight_decay' 
    REPO_DIR    = Path('/kaggle/working/SeniorProject')
    if not REPO_DIR.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    MODELS_DIR     = REPO_DIR / 'models'
    NOTEBOOK_DIR   = MODELS_DIR / 'notebooks' / 'fundamental'
    OHLCV_DATA_DIR = Path('/kaggle/input/datasets/kingz101/sp500-daily-raw')
    FUND_DATA_DIR  = Path('/kaggle/input/sp500-quarterly-fundamental')
    CHECKPOINT_DIR = Path('/kaggle/working/checkpoint/patchtst_fund_cls')
    SAVE_DIR       = Path('/kaggle/working/save_dir_fund')
else:
    NOTEBOOK_DIR = Path.cwd()
    if NOTEBOOK_DIR.name != 'fundamental':
        NOTEBOOK_DIR = Path('models/notebooks/fundamental').resolve()
    MODELS_DIR     = NOTEBOOK_DIR.parents[1]   # models/
    _local_data    = MODELS_DIR / 'data_raw'
    OHLCV_DATA_DIR = _local_data
    FUND_DATA_DIR  = _local_data
    CHECKPOINT_DIR = NOTEBOOK_DIR / 'checkpoint' / 'patchtst_fund_cls'
    SAVE_DIR       = NOTEBOOK_DIR / 'save_dir_fund'

sys.path.insert(0, str(MODELS_DIR))

from patchtst_lib.classification_head import PatchTSTClassifier
from patchtst_lib.fundamental.config import make_fundamental_patchtst_config
from patchtst_lib.fundamental.dataset import (
    QuarterlyFundamentalDataset,
    compute_fundamental_class_weights,
    split_fundamentals_by_date,
    summarize_fundamental_dataset,
)
from patchtst_lib.fundamental.features import (
    FUND_FEATURE_COLUMNS,
    build_feature_df,
    load_raw_fundamentals,
)
from patchtst_lib.labeling import CLASS_NAMES, LabelConfig
from patchtst_lib.training import compute_metrics

sns.set_theme(style='whitegrid')


Cloning into '/kaggle/working/SeniorProject'...


In [3]:
from pathlib import Path
for p in Path('/kaggle/input').iterdir():
    print(p)
    if p.is_dir():
        for f in p.iterdir():
            print(' ', f)

/kaggle/input/datasets
  /kaggle/input/datasets/kingz101
/kaggle/input/sp500-quarterly-fundamental
  /kaggle/input/sp500-quarterly-fundamental/quarterly_fundamentals.csv


In [4]:
# ============================================================
# Hyper-parameter config — edit here, nowhere else.
# ============================================================

# --- Data ---
FUND_CSV_FILENAME  = 'quarterly_fundamentals.csv'
OHLCV_CSV_FILENAME = 'historic_data_rows.csv'

# Chronological split cutoffs.
TRAIN_END = '2019-12-31'
VAL_END   = '2021-12-31'
# Test = everything after VAL_END.

# --- Model ---
CONTEXT_LENGTH = 12   # quarters
PATCH_LENGTH   = 2
PATCH_STRIDE   = 1
D_MODEL        = 32   # bump to 64 if loss plateaus
N_HEADS        = 4
N_LAYERS       = 2
HEAD_DROPOUT   = 0.3
ATTN_DROPOUT   = 0.2
HORIZON        = 1    # single next-quarter prediction
N_CLASSES      = 3

# --- Labels ---
LABEL_RULE      = 'fixed_pct'
LABEL_THRESHOLD = 0.05      # ±5% quarterly return → up / down
PUBLISH_LAG_DAYS  = 45
FORECAST_LAG_DAYS = 63

# --- Training ---
BATCH_SIZE     = 64
MAX_EPOCHS     = 2
LR             = 2e-4
WEIGHT_DECAY   = 0.05
WARMUP_RATIO   = 0.05
GRAD_CLIP      = 1.0
EARLY_STOP_PATIENCE = 10
SEED = 42

print(f'CONTEXT_LENGTH={CONTEXT_LENGTH} | PATCH_LENGTH={PATCH_LENGTH} | '
      f'D_MODEL={D_MODEL} | HORIZON={HORIZON} | BATCH={BATCH_SIZE}')


CONTEXT_LENGTH=12 | PATCH_LENGTH=2 | D_MODEL=32 | HORIZON=1 | BATCH=64


In [5]:
import re

_OHLCV_CANONICAL = {
    'date': 'Date', 'ticker': 'Ticker', 'open': 'Open',
    'high': 'High', 'low': 'Low', 'close': 'Close',
    'volume': 'Volume', 'sector': 'Sector',
}

def _normalize_ohlcv_cols(df):
    rename = {c: _OHLCV_CANONICAL[c.strip().lower()]
              for c in df.columns if c.strip().lower() in _OHLCV_CANONICAL}
    df = df.rename(columns=rename)
    df['Date'] = pd.to_datetime(df['Date'])
    return df

# Load fundamentals.
fund_csv = FUND_DATA_DIR / FUND_CSV_FILENAME
if not fund_csv.exists():
    raise FileNotFoundError(
        f'{fund_csv} not found.  On Kaggle: attach dataset '
        f'kingz101/sp500-quarterly-fundamental (file: {FUND_CSV_FILENAME}).'
    )
raw_fund = load_raw_fundamentals(str(fund_csv))
feat_df  = build_feature_df(raw_fund)
print(f'Feature DataFrame: {feat_df.shape} | Tickers: {feat_df["Ticker"].nunique()}')
print(f'Date range: {feat_df["datadate"].min().date()} -> {feat_df["datadate"].max().date()}')

# Load OHLCV prices (used only for labels inside the dataset).
ohlcv_csv = OHLCV_DATA_DIR / OHLCV_CSV_FILENAME
if not ohlcv_csv.exists():
    raise FileNotFoundError(
        f'{ohlcv_csv} not found.  On Kaggle: attach dataset kingz101/sp500-daily-raw.'
    )
prices_df = _normalize_ohlcv_cols(pd.read_csv(ohlcv_csv))
print(f'OHLCV DataFrame: {prices_df.shape}')


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4781: RuntimeWarning: invalid value encountered in add
  lerp_interpolation = asanyarray(add(a, diff_b_a * t, out=out))
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning:

Feature DataFrame: (31582, 13) | Tickers: 503
Date range: 2010-01-31 -> 2026-04-30
OHLCV DataFrame: (2922054, 8)


In [6]:
label_config = LabelConfig(
    rule=LABEL_RULE,
    threshold=LABEL_THRESHOLD,
)

train_feat, val_feat, test_feat = split_fundamentals_by_date(
    feat_df,
    train_end=TRAIN_END,
    val_end=VAL_END,
    context_length=CONTEXT_LENGTH,
)
print(f'Split sizes: train={len(train_feat)} val={len(val_feat)} test={len(test_feat)} rows')

dataset_kwargs = dict(
    feature_columns=FUND_FEATURE_COLUMNS,
    context_length=CONTEXT_LENGTH,
    label_config=label_config,
    publish_lag_days=PUBLISH_LAG_DAYS,
    forecast_lag_days=FORECAST_LAG_DAYS,
)

train_ds = QuarterlyFundamentalDataset(train_feat, prices_df, **dataset_kwargs)
val_ds   = QuarterlyFundamentalDataset(val_feat,   prices_df, **dataset_kwargs)
test_ds  = QuarterlyFundamentalDataset(test_feat,  prices_df, **dataset_kwargs)

print()
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    print(summarize_fundamental_dataset(name, ds).to_string(index=False))

if train_ds.skipped_tickers:
    print(f'\nSkipped tickers (no OHLCV match): {train_ds.skipped_tickers}')


Split sizes: train=19066 val=9855 test=14369 rows

dataset  windows  down  flat   up  down_pct  flat_pct   up_pct
  train    13163  2195  6044 4924  0.166755  0.459166 0.374079
dataset  windows  down  flat   up  down_pct  flat_pct   up_pct
    val     3911   996  1241 1674  0.254666   0.31731 0.428024
dataset  windows  down  flat   up  down_pct  flat_pct   up_pct
   test     8415  2403  3226 2786  0.285561  0.383363 0.331075

Skipped tickers (no OHLCV match): ['ABNB', 'APP', 'CARR', 'CEG', 'CRWD', 'CTVA', 'DASH', 'DDOG', 'HOOD', 'OTIS', 'PLTR', 'Q', 'UBER', 'VICI']


In [7]:
class_weights = compute_fundamental_class_weights(train_ds)
print(f'Class weights: {class_weights.tolist()}')

patchtst_cfg = make_fundamental_patchtst_config(
    context_length=CONTEXT_LENGTH,
    patch_length=PATCH_LENGTH,
    patch_stride=PATCH_STRIDE,
    d_model=D_MODEL,
    num_attention_heads=N_HEADS,
    num_hidden_layers=N_LAYERS,
    head_dropout=HEAD_DROPOUT,
    attention_dropout=ATTN_DROPOUT,
)

model = PatchTSTClassifier(
    config=patchtst_cfg,
    horizon=HORIZON,
    n_classes=N_CLASSES,
    class_weights=class_weights,
)
model = model.to(device=device, dtype=dtype)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

# Quick sanity forward pass.
_dummy = next(iter(DataLoader(train_ds, batch_size=2)))
_out = model(
    past_values=_dummy['past_values'].to(device=device, dtype=dtype),
    labels=_dummy['labels'].to(device),
)
print(f'Logits shape: {_out.logits.shape}  Loss: {_out.loss.item():.4f}')


Class weights: [1.6584243774414062, 0.602290153503418, 0.7392854690551758]
Model parameters: 37,475
Logits shape: torch.Size([2, 1, 3])  Loss: 1.2039


In [8]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=GRAD_CLIP,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_dir=str(CHECKPOINT_DIR / 'logs'),
    logging_steps=50,
    dataloader_num_workers=NUM_WORKERS,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    seed=SEED,
    report_to='tensorboard',
)

early_stop = EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[early_stop],
)

print(f'Starting training | max epochs={MAX_EPOCHS} | early stop patience={EARLY_STOP_PATIENCE}')
train_result = trainer.train()
print('Training complete.')
print(train_result.metrics)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
2026-05-18 23:11:03.912231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779145864.103005      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779145864.156519      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779145864.591262      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779145864.591308      22 computation_placer.cc:177] computatio

Starting training | max epochs=2 | early stop patience=10


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


AttributeError: 'tuple' object has no attribute 'reshape'

In [ ]:
model.eval()
all_logits, all_labels = [], []
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
import torch

with torch.no_grad():
    for batch in test_loader:
        pv = batch['past_values'].to(device=device, dtype=dtype)
        out = model(past_values=pv)
        all_logits.append(out.logits.detach().cpu().float().numpy())
        all_labels.append(batch['labels'].numpy())

logits_np = np.concatenate(all_logits, axis=0)   # (N, 1, 3)
labels_np = np.concatenate(all_labels, axis=0)   # (N, 1)
preds_np  = np.argmax(logits_np[:, 0, :], axis=-1)
labels_flat = labels_np[:, 0]

acc  = accuracy_score(labels_flat, preds_np)
mf1  = f1_score(labels_flat, preds_np, average='macro', zero_division=0)
print(f'Test accuracy: {acc:.4f}  |  Macro-F1: {mf1:.4f}')
print()
print(classification_report(labels_flat, preds_np, target_names=['down', 'flat', 'up'], zero_division=0))

# Confusion matrix.
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
cm = confusion_matrix(labels_flat, preds_np)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['down', 'flat', 'up']).plot(ax=ax, colorbar=False)
ax.set_title('Fundamental Model — Test Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
import json as _json

# Save model.
model.patchtst.save_pretrained(str(SAVE_DIR / 'patchtst_fund'))

# Save training metadata.
metadata = {
    'saved_at': datetime.utcnow().isoformat() + 'Z',
    'model_type': 'fundamental',
    'context_length': CONTEXT_LENGTH,
    'patch_length': PATCH_LENGTH,
    'patch_stride': PATCH_STRIDE,
    'd_model': D_MODEL,
    'num_hidden_layers': N_LAYERS,
    'num_attention_heads': N_HEADS,
    'horizon': HORIZON,
    'n_classes': N_CLASSES,
    'feature_columns': FUND_FEATURE_COLUMNS,
    'label_rule': LABEL_RULE,
    'label_threshold': LABEL_THRESHOLD,
    'publish_lag_days': PUBLISH_LAG_DAYS,
    'forecast_lag_days': FORECAST_LAG_DAYS,
    'train_end': TRAIN_END,
    'val_end': VAL_END,
    'train_windows': len(train_ds),
    'val_windows': len(val_ds),
    'test_windows': len(test_ds),
    'test_accuracy': float(acc),
    'test_macro_f1': float(mf1),
    'class_weights': class_weights.tolist(),
    'skipped_tickers': train_ds.skipped_tickers,
}

meta_path = SAVE_DIR / 'fund_training_metadata.json'
with open(meta_path, 'w') as f:
    _json.dump(metadata, f, indent=2)

print(f'Model saved to {SAVE_DIR / "patchtst_fund"}')
print(f'Metadata saved to {meta_path}')
